In [ ]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.5/819.5 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 12.0 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [ ]:
import os
from pyngrok import ngrok

In [ ]:
ngrok.kill()

In [ ]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)



Ngrok URL: https://amigo-capable-untying.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://amigo-capable-untying.ngrok-free.dev


True

In [ ]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

google_search_tool = Tool(
   google_search=GoogleSearch() #利用搜尋引擎，找到最新資訊
)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        system_instruction="你是一個中文的AI助手，請用繁體中文回答",
        tools=[google_search_tool],
        response_modalities=["TEXT"],
    )
)

In [ ]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [ ]:
result = stateful_query("簡介明新科技大學")
print(result)

明新科技大學（Minghsin University of Science and Technology），簡稱明新科大，是一所位於臺灣新竹縣新豐鄉的私立科技大學。學校於1966年以「明新工業專科學校」創校。歷經發展，於1997年改制為「明新技術學院」，並最終在2002年9月獲教育部核准升格為「明新科技大學」。

明新科大秉持「堅毅、求新、創造」的校訓，其校名「明新」取自《大學》「在明明德，在新民，在止於至善」的精義，旨在闡揚人類德性與情操，並期許學子具備專業學問、優良技術，以達全人發展的境界。學校願景為「深耕在地、放眼國際」，教育目標是「培養具實務經驗與人文素養之專業人才」。

目前，明新科技大學設有半導體學院、工程學院、管理學院、民生學院、人文與設計學院、共同教育學院等六大學院，涵蓋多個學系、學位學程及碩士班。學校積極發展多元學習、全球視野、永續經營與技術創新等四大育才特色，特別在半導體、AI、元宇宙、風電綠能等前瞻產業領域深耕，致力於培養業界所需人才。

明新科大尤其著重產學合作，與新竹科學園區、新竹工業區等在地產業鏈結緊密，被譽為「產業人才轉運站」。學校投資兩億元打造「半導體基地」，設置半導體封裝測試類產線，並設有技職體系中第一座「半導體學院」，以培育半導體產業設備、維修、封裝、測試等實務人才。根據1111人力銀行統計，明新科大在半導體產業界最愛聘用的畢業生中名列前茅，是唯一入榜的私立科大。此外，學校也致力於推動國際化，擁有全國居冠的國際學生人數，並與多國學術單位簽訂合作計畫，提升學生的國際移動力與全球學習視野。


In [ ]:
result2 = stateful_query("校長是誰？")
print(result2)

明新科技大學現任校長為**呂明峯教授**。

他於2025年2月1日正式上任， 並於2025年1月16日舉行了布達暨交接典禮。 呂明峯校長在明新科大服務長達34年，擁有豐富的業界經驗與學術涵養，並曾主導打造全台首座半導體封裝測試類產線，並推動成立半導體學院。 他提出「四大核心模組」的校務治理策略，目標是將明新科大打造成「具國際魅力的產業科技大學」。


In [ ]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    text = event.message.text
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        if text.startswith('AI '):
            prompt = text[3:]
            reply_text = stateful_query(prompt)
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )

        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit


BODY:  {"destination":"U712d2d53b4bcdc15fe49feed703ea8a8","events":[{"type":"message","message":{"type":"text","id":"615035004355084705","quoteToken":"RQlA5SC-g6UTAI-kgbp5kWMBdJGBXIcTnxLjVlvj1XWZOqeU2QI1d2xPPhLLs-JgA_YBnnN06_-sOOd6okCAzRglj8Qxc5ple8fpljCd1W7huFOv9QKuQd0ElgX7uWOhn_OROYNnopY7sQnEQEJBrg","markAsReadToken":"ejzSAkQ6Qa9aj-2hjVaABdYOCIgDhglKRzu6e3he-Z_c-IQh4P-SRTiaka4AFlwTX9Go60v8fdOodldzjFSc-zdwUUMOaAKBy_NFDvrltY_zPe6B5TyJAwc-l6nqV11oFEaKUItn4WMgVYqCdGUYEwyBzZ-NkOiJLKU9qAvxNFcRhxEsQp3YG8NP3UExcOCgtqrhSqVYyiBQce-0JenHBg","text":"介紹明新科大，限100字"},"webhookEventId":"01KS6VV8HBXF9KS63C81X5QYE5","deliveryContext":{"isRedelivery":false},"timestamp":1779420733603,"source":{"type":"user","userId":"U2a89fc3db06801f30d432e1507e0ee19"},"replyToken":"769c15bfac504d83bd6754a26886b11d","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [22/May/2026 03:32:14] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"U712d2d53b4bcdc15fe49feed703ea8a8","events":[{"type":"message","message":{"type":"text","id":"615035025963876474","quoteToken":"V4QEBuY4afLMcKg6Ud9fyNpqzC7nDm6pz6qHgXEznrVVUeABT269zxF1FI-QDq6dmaQQ8HTtwbYgdc0srm_MBLi01EH9CvYV89f8f5QpECEH645dGcPuAid4S3DxIuZqyCH1kBswHSDHu5YlB4no2g","markAsReadToken":"A9Sb_rPAxM0I5nGJYj6pbdZxAwqfaunF8sZmga96r5lUBZSb0mrEz2KGBIiaZj7KHWeih8EPrGNiv8W00Wo1iFpGzQioF6-JG4a3Bp5TI7m0sxEy0NMuVnIuLo_5nPWovdaY4h0UnNCxbxybw8j1Qy3JLYyOvpdroSRoNklTZ4KF5hvp4nZkBoeZQrz11sf0ttcnsZHlBD25-TdaMckJHg","text":"AI 介紹明新科大，限100字"},"webhookEventId":"01KS6VVN2F3AXXJAJBC558RSXB","deliveryContext":{"isRedelivery":false},"timestamp":1779420746559,"source":{"type":"user","userId":"U2a89fc3db06801f30d432e1507e0ee19"},"replyToken":"f2f93fecb036414a80e3921f83e4e715","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [22/May/2026 03:32:29] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"U712d2d53b4bcdc15fe49feed703ea8a8","events":[{"type":"message","message":{"type":"text","id":"615035048042954893","quoteToken":"YreZrd5YC-x6Uk_PekTspRvI1h3d-VD_da_7C4zz7TyknjfTsSex-7ZODWglejUkHLwanc6vvYDCBAVeRULeeURaytC1LhVK8g1YHibH_Ee1u3ZIgqI_sqvIMlbJfhDAJVHS0mpGchmnWQTorrMo1A","markAsReadToken":"Dxz2G0ps-gyoVhl-TQqLn0beQ1S6EkoqGA1z41v6npSE6P1dDzuHpjSgCLPqT4r0G5quL9dTx37c6LL0kwWqbXRJZap4R_I4Q6HfHp8pxjo5MYvc--qNckBaSu11mhAuIDAab5iiIIvuDS28T08JwDcDi4Nw5iYHQ-37R-Q5w186yaNg2NIg73qIUrPM36kSLZS878f5NmtZ0XWIBZwhXw","text":"校長是誰"},"webhookEventId":"01KS6VW1TADGZBV31ZEJK846JA","deliveryContext":{"isRedelivery":false},"timestamp":1779420759640,"source":{"type":"user","userId":"U2a89fc3db06801f30d432e1507e0ee19"},"replyToken":"548302bc5a52493795eb4ebe3dc81b10","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [22/May/2026 03:32:40] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"U712d2d53b4bcdc15fe49feed703ea8a8","events":[{"type":"message","message":{"type":"text","id":"615035061112668182","quoteToken":"bbDUVOeyY5ryfqg62GUqnGuW7sOLufXFKj5W37V49uzGrHvhvabnB6-fNqkYyeA_uUWR0rPyZC2paTO8MRdUcgSpcuKwOXPhKBGHzFfBB9D2eeZFJh5ATtFpmjCUbQ57TbriSkXHXnB8AJbSOLoUhg","markAsReadToken":"dsQ6KwjKo1Vwb6PQe2JSklShATIRo9CQW7B0XmPTUB6XvctAdSD-UfrrYKb5h-vqt63D0wYJsOeWscGaFJaJJOPwx7sv7cLu-FnD1qa9K_PKr7FoikCyT1uSb-eptm8ZPH8Ulk__pqWrj09w9Jfz_14ZlaafVA0eurzawj8cGXk6FfEEKnNIl35RFQ9gmuDTP1WfscwdJT-LUYeEjnxxuQ","text":"AI 校長是誰"},"webhookEventId":"01KS6VW9X5W2F1K62ABZR8DYHD","deliveryContext":{"isRedelivery":false},"timestamp":1779420767581,"source":{"type":"user","userId":"U2a89fc3db06801f30d432e1507e0ee19"},"replyToken":"3c3f2688d9dd415f8d58f2a4e4862c5d","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [22/May/2026 03:32:49] "POST / HTTP/1.1" 200 -
